In [3]:
import pandas as pd
from pathlib import Path

# -----------------------------
# 0) 경로 설정 (너 구조 기준)
# -----------------------------
PROJECT_ROOT = Path("/Users/gimgyumin/Documents/commercial-area-analysis-ai")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
MAP_DIR = DATA_DIR / "category_maps"
OUT_DIR = PROJECT_ROOT / "ai" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SALES_PATH = RAW_DIR / "sales_info" / "서울시(추정매출-행정동).csv"
STORE_PATH = RAW_DIR / "store_info" / "상가(상권)정보_서울.csv"       # 파일명 다르면 수정
POP_PATH   = RAW_DIR / "population_info" / "서울시(유동인구-행정동).csv"

SALES_MAP_PATH = MAP_DIR / "sales_category_map.csv"
STORE_MAP_PATH = MAP_DIR / "store_category_map.csv"

OUT_PATH = OUT_DIR / "merged_3sources_by_dong_qtr_category.csv"


# -----------------------------
# 1) 한글 CSV 안전 로더
# -----------------------------
def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


# -----------------------------
# 2) 매핑 테이블 로드
# -----------------------------
sales_map_df = read_csv_kor(SALES_MAP_PATH)
store_map_df = read_csv_kor(STORE_MAP_PATH)

SALES_CATEGORY_MAP = dict(zip(sales_map_df["서비스_업종_코드_명"], sales_map_df["통합_카테고리"]))
STORE_CATEGORY_MAP = dict(zip(store_map_df["상권업종중분류명"], store_map_df["통합_카테고리"]))


# -----------------------------
# 3) 원본 로드
# -----------------------------
sales_df = read_csv_kor(SALES_PATH)
store_df = read_csv_kor(STORE_PATH)
pop_df   = read_csv_kor(POP_PATH)


# -----------------------------
# 4) 컬럼명/타입 통일
# -----------------------------
# sales/pop: 행정동_코드 -> 행정동코드
sales_df = sales_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})
pop_df   = pop_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})

# store는 이미 행정동코드, 행정동명 있음

# 타입 통일
for df in (sales_df, store_df, pop_df):
    df["행정동코드"] = pd.to_numeric(df["행정동코드"], errors="coerce").astype("Int64")

# 키 컬럼 체크
required_sales = ["기준_년분기_코드", "행정동코드", "서비스_업종_코드_명", "당월_매출_금액"]
for c in required_sales:
    if c not in sales_df.columns:
        raise KeyError(f"sales_df에 {c} 없음")

required_pop = ["기준_년분기_코드", "행정동코드", "총_유동인구_수"]
for c in required_pop:
    if c not in pop_df.columns:
        raise KeyError(f"pop_df에 {c} 없음")

required_store = ["행정동코드", "상권업종중분류명"]
for c in required_store:
    if c not in store_df.columns:
        raise KeyError(f"store_df에 {c} 없음")


# -----------------------------
# 5) SALES: 통합카테고리 붙이고 분기/행정동/카테고리로 집계
# -----------------------------
sales_df["통합_카테고리"] = sales_df["서비스_업종_코드_명"].map(SALES_CATEGORY_MAP)
sales_df = sales_df.dropna(subset=["통합_카테고리"]).copy()

sales_agg = (
    sales_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)["당월_매출_금액"]
    .agg(sales_sum="sum", sales_mean="mean", sales_count="count")
    .reset_index()
)


# -----------------------------
# 6) STORE: 통합카테고리 붙이고 행정동/카테고리로 점포수 집계
# -----------------------------
store_df["통합_카테고리"] = store_df["상권업종중분류명"].map(STORE_CATEGORY_MAP)
store_df = store_df.dropna(subset=["통합_카테고리"]).copy()

store_agg = (
    store_df.groupby(["행정동코드", "통합_카테고리"], dropna=False)
    .size()
    .reset_index(name="store_count")
)

# (선택) 행정동명도 붙여두면 보기 편함
dong_name = store_df[["행정동코드", "행정동명"]].drop_duplicates()
store_agg = pd.merge(store_agg, dong_name, on="행정동코드", how="left")


# -----------------------------
# 7) POP: 분기/행정동 단위로 집계 (총유동인구)
# -----------------------------
pop_agg = (
    pop_df.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)["총_유동인구_수"]
    .agg(pop_sum="sum", pop_mean="mean", pop_count="count")
    .reset_index()
)

# (선택) 행정동명 붙이기
pop_name = pop_df[["행정동코드", "행정동명"]].drop_duplicates()
pop_agg = pd.merge(pop_agg, pop_name, on="행정동코드", how="left")


# -----------------------------
# 8) MERGE: sales(기준) + store + pop
# -----------------------------
merged = pd.merge(
    sales_agg,
    store_agg[["행정동코드", "통합_카테고리", "store_count"]],
    on=["행정동코드", "통합_카테고리"],
    how="left"
)

merged = pd.merge(
    merged,
    pop_agg[["기준_년분기_코드", "행정동코드", "pop_sum", "pop_mean", "pop_count"]],
    on=["기준_년분기_코드", "행정동코드"],
    how="left"
)

# 행정동명은 한 군데에서만 정리해서 붙임(중복 방지)
merged = pd.merge(
    merged,
    pop_name,  # pop에서 만든 행정동코드-명 테이블
    on="행정동코드",
    how="left"
)

# -----------------------------
# 9) 결측 처리
# -----------------------------
merged["store_count"] = merged["store_count"].fillna(0).astype(int)
for c in ["pop_sum", "pop_mean", "pop_count"]:
    merged[c] = merged[c].fillna(0)

# -----------------------------
# 10) 저장
# -----------------------------
merged.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("✅ 저장 완료:", OUT_PATH)
print("행 수:", len(merged))
print("컬럼 수:", merged.shape[1])
print("샘플:")
print(merged.head(3))

✅ 저장 완료: /Users/gimgyumin/Documents/commercial-area-analysis-ai/ai/outputs/merged_3sources_by_dong_qtr_category.csv
행 수: 27163
컬럼 수: 11
샘플:
   기준_년분기_코드     행정동코드  통합_카테고리  sales_sum   sales_mean  sales_count  \
0      20251  11110515  B2B 서비스  268180168  134090084.0            2   
1      20251  11110515      미용실  261655455  261655455.0            1   
2      20251  11110515    분식/간식  876235565  438117782.5            2   

   store_count  pop_sum   pop_mean  pop_count   행정동명  
0          251  3322244  3322244.0          1  청운효자동  
1           24  3322244  3322244.0          1  청운효자동  
2           47  3322244  3322244.0          1  청운효자동  


기준_년분기_코드,
행정동코드,
통합_카테고리,
sales_sum,
sales_mean,
sales_count,
store_count,
pop_sum,
pop_mean,
pop_count,
행정동명


In [4]:
print(merged.columns)

Index(['기준_년분기_코드', '행정동코드', '통합_카테고리', 'sales_sum', 'sales_mean',
       'sales_count', 'store_count', 'pop_sum', 'pop_mean', 'pop_count',
       '행정동명'],
      dtype='str')
